# Phase 2 — Validation thresholds and error analysis

Run `python scripts/phase_02_acceptance.py` first. Threshold maximizes exact validation F2; equal maxima choose the highest threshold. Models are proposed by validation AP, then F2, then name. This notebook displays aggregate validation slices, never raw FP/FN examples or test predictions.

In [ ]:
import json
import pandas as pd
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython import get_ipython
from matplotlib_inline.backend_inline import configure_inline_support
configure_inline_support(get_ipython(), 'module://matplotlib_inline.backend_inline')
from fraud_detection.paths import find_project_root
from fraud_detection.pipelines.train import load_verified_results

ROOT = find_project_root()
OUTPUT = ROOT / 'artifacts/phase_02'
result = load_verified_results(ROOT)
print('Source kind:', result['source_kind'])
print('Config SHA:', result['config_sha256'])
print('Test opened:', result['test_opened'])


In [ ]:
proposal = result['candidate']
print('Proposal:', proposal['model_name'], proposal['status'])
print('Decision SHA:', proposal['decision_sha256'])
run = next(run for run in result['runs'] if run['run_id'] == proposal['run_id'])
display(pd.DataFrame([run['validation_metrics']]))
operating_points = pd.DataFrame(run['threshold_summary'])
display(operating_points[['threshold', 'precision', 'recall', 'f2', 'alerts_per_1000', 'fp', 'fn']])
ax = operating_points.plot(x='threshold', y=['precision', 'recall', 'f2'], figsize=(9, 4))
ax.set_title('Validation operating points; exact search selects the saved threshold')
plt.tight_layout()
plt.show()
for field, slices in run['error_slices'].items():
    print('Validation error slices:', field)
    display(pd.DataFrame(slices))
print('Test remains sealed:', not result['test_opened'])
print('Reviewers:', proposal['reviewers'])


## Decision gate

Human reviewers must inspect the exact run ID, decision hash, source and split hashes, limitations and alert trade-off. Empty reviewer fields are pending, not approval. No final test evaluation or production-readiness claim is made in Phase 2. Time has only a two-day source window; PCA components cannot be assigned invented business meanings.